# Mapping Bright Stars in DP2

This notebook demonstrates how to map out all the stars in DP2.

For rapidly getting up to speed, the catalog queried from the server has already been made available, but the full query is also demonstrated.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy.units as units
from lsst.rsp import get_tap_service, retrieve_query
from astropy.coordinates import SkyCoord
import skyproj
from dustmaps.sfd import SFDQuery
import hpgeom

The following section will set up the Table Access Protocol (TAP) database access to the data

In [ ]:
service = get_tap_service("tap")
query = 'SELECT * FROM tap_schema.schemas'
results = service.search(query)
results_table = service.search(query).to_table()
print("These are the schemas available in the TAP service:")
print(results_table)

In [ ]:
query = "SELECT * FROM tap_schema.tables " \
        "WHERE tap_schema.tables.schema_name = 'dp2'" \
        "ORDER BY table_index ASC"
results = service.search(query).to_table()
print("These are the data tables available from the dp2 schema:")
print(results)

In [ ]:
query = "SELECT column_name, datatype, description, unit " \
        "FROM tap_schema.columns " \
        "WHERE table_name = 'dp2.Object'"
results = service.search(query).to_table()
print("These are the columns available in the Object (coadd catalog) table:")
for cname in results['column_name']:
    print(cname)

In [ ]:
# Set do_query = True if you want to execute the full query

results = None
do_query = False

The following cell will do a big query.  This takes a long time so is commented out and the data can be loaded below.

This query converts the PSF fluxes to magnitudes, gets two values for the "extendendess" for star/galaxy separation, and finally does a very bright selection of mag_psf_i < 18.0.

In [ ]:
if do_query:
    query = "SELECT coord_ra, coord_dec, refExtendedness, griz_model_extendedness,"\
            "scisql_nanojanskyToAbMag(g_psfFlux) as gmag, scisql_nanojanskyToAbMagSigma(g_psfFlux, g_psfFluxErr) as gmagErr, g_psfFlux_flag, "\
            "scisql_nanojanskyToAbMag(r_psfFlux) as rmag, scisql_nanojanskyToAbMagSigma(r_psfFlux, r_psfFluxErr) as rmagErr, r_psfFlux_flag, "\
            "scisql_nanojanskyToAbMag(i_psfFlux) as imag, scisql_nanojanskyToAbMagSigma(i_psfFlux, i_psfFluxErr) as imagErr, i_psfFlux_flag, "\
            "scisql_nanojanskyToAbMag(z_psfFlux) as zmag, scisql_nanojanskyToAbMagSigma(z_psfFlux, z_psfFluxErr) as zmagErr, z_psfFlux_flag "\
            "FROM dp2.Object "\
            "WHERE scisql_nanojanskyToAbMag(i_psfFlux) < 18.0 AND scisql_nanojanskyToAbMagSigma(i_psfFlux, i_psfFluxErr) < 0.2 AND i_pixelFlags_saturated=0"
    job = service.submit_job(query)
    print('Job URL is', job.url)
    print('Pre-Job phase is', job.phase)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    print('Job phase is', job.phase)
    results = job.fetch_result().to_table()

In [ ]:
if results is None:
    results = Table.read("/home/erykoff/data/bright_object_psf_mags.fits")
print("Loaded ", len(results), " rows.")
print(results)

In [ ]:
# Select objects that have finite griz psf magnitudes
use = np.isfinite(results["gmag"]) & np.isfinite(results["rmag"]) & np.isfinite(results["imag"]) & np.isfinite(results["zmag"])
results = results[use]
print("There are ", len(results), "objects with finite griz magnitudes.")

In [ ]:
# Select out the colors and positions
gmr = results["gmag"] - results["rmag"]
rmi = results["rmag"] - results["imag"]
imz = results["imag"] - results["zmag"]
ra = results["coord_ra"]
dec = results["coord_dec"]

# Perils of star/galaxy separation

This section we're going to look at the colors of these bright objects (mag_i < 18) over the full DP2 survey. We are going to try two star/galaxy separator options:

* `refExtendendess` is a measure of the "extendedness" of objects in the reference band for the object. Typically this is i band, but may be r (and so forth) if the object isn't observed in i.  Since we are requiring bright i observations, these will all be i-band reference. The extendedness is based on a comparison of a model fit to the object and a PSF fit.  If they are sufficiently different in measured flux then it is extended (`refExtendedness == 1.0`).  If they are similar then it is not extended (`refExtendedness == 0.0`).
* `griz_model_extendedness` is derived from a Sersic model fit to all of griz simultaneously.  It is based on a continuous likelihood and therefore ranges from 0 to 1.  The recommended threshold between stars and galaxies that works in well-measured fields is 0.2.  Note that this will only work for objects that have been detected in all of griz (like we have here).

Now let us look at some aspects of the full DP2 catalog.
We will start with a color/color diagram.

As these are all bright objects, we are dominated by stars.

In [ ]:
plt.hexbin(rmi, gmr, bins="log", extent=[0, 4, 0, 5])
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Bright i<18 DP2 objects")
plt.colorbar(label="Density")
plt.show()

There are a number of interesting features here.
* To clarify, up and right is "redder", down and left is "bluer"
* There is a bunch of junk with 0 < r-i < 0.5 (note the log density scale); ignore it
* The density of blue objects (stars) is much higher than red
* There is a turnoff at g-r ~1.4 and another locus that continues up.  Why?

In [ ]:
# Try two different star/galaxy separations

stars1 = (results["refExtendedness"] < 0.5)
galaxies1 = (results["refExtendedness"] > 0.5)
stars2 = (results["griz_model_extendedness"] < 0.2)
galaxies2 = (results["griz_model_extendedness"] > 0.8)

print("Stars/galaxies using baseline refExtendedness")
print(stars1.sum(), galaxies1.sum())
print("Stars/galaxies using griz_model_extendedness")
print(stars2.sum(), galaxies2.sum())

In [ ]:
plt.hexbin(rmi[stars1], gmr[stars1], bins="log", extent=[0, 4, 0, 5])
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Bright i<18 DP2 \"stars\" (1)")
plt.colorbar(label="Density")
plt.show()

plt.hexbin(rmi[galaxies1], gmr[galaxies1], bins="log", extent=[0, 4, 0, 5])
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Bright i<18 DP2 \"galaxies\" (1)")
plt.colorbar(label="Density")
plt.show()

In [ ]:
plt.hexbin(rmi[stars2], gmr[stars2], bins="log", extent=[0, 4, 0, 5])
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Bright i<18 DP2 \"stars\" (2)")
plt.colorbar(label="Density")
plt.show()

plt.hexbin(rmi[galaxies2], gmr[galaxies2], bins="log", extent=[0, 4, 0, 5])
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Bright i<18 DP2 \"galaxies\" (2)")
plt.colorbar(label="Density")
plt.show()

Neither of the selections above does a good job of selecting out "galaxies".  These objects are overwhelmingly stars, and apparently our PSF models are not fantastic at the very bright end.

Nevertheless, the extra locus isn't galaxies.

Now we can look at this plot split by position.  Maybe it's the Milky Way galaxy?

In [ ]:
pos = SkyCoord(ra, dec, frame="icrs")
lb = pos.galactic
galactic_longitude = lb.l.degree
galactic_latitude = lb.b.degree

In [ ]:
high_lat_stars = (np.abs(galactic_latitude[stars1]) > 30.0)

In [ ]:
plt.hexbin(rmi[stars1][high_lat_stars], gmr[stars1][high_lat_stars], bins="log", extent=[0, 4, 0, 5])
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Bright i<18 DP2 \"stars\" (1) (High Latitude)")
plt.colorbar(label="Density")
plt.show()


Now we're getting somewhere!  This is a nice clean stellar locus, with some excess junk but it looks nice zoomed in:

In [ ]:
plt.hexbin(rmi[stars1][high_lat_stars], gmr[stars1][high_lat_stars], bins="log", extent=[0, 2.5, 0, 1.7])
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Bright i<18 DP2 \"stars\" (1) (High Latitude)")
plt.colorbar(label="Density")
plt.show()


# Mapping Out Positions

In this next section we use `skyproj` to make maps of where the stars are, their average colors, etc.

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

sp = skyproj.McBrydeSkyproj(ax=ax)
# We are zooming in on the region with the main part of the DP2 catalog.
sp.draw_hpxbin(ra[stars1], dec[stars1], nest=True, lon_range=[-130, 20], lat_range=[-60, 15])
sp.draw_milky_way()
sp.draw_colorbar(label="Star Density")
plt.show()

# Replot with log normalization of density
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

sp = skyproj.McBrydeSkyproj(ax=ax)
# We are zooming in on the region with the main part of the DP2 catalog.
sp.draw_hpxbin(ra[stars1], dec[stars1], nest=True, lon_range=[-130, 20], lat_range=[-60, 15], norm="log")
sp.draw_milky_way()
sp.draw_colorbar(label="Star Density")
plt.show()

Two interesting things can be cleaned from this map:
* The density rises up as we get to lower Galactic latitude (black+dashed lines)
* But then it drops down into the Galactic plane, with lots of gaps.

Why?

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

sp = skyproj.McBrydeSkyproj(ax=ax)
# We are zooming in on the region with the main part of the DP2 catalog.
sp.draw_hpxbin(
    ra[stars1], 
    dec[stars1], 
    nest=True, 
    lon_range=[-130, 20], 
    lat_range=[-60, 15], 
    C=gmr[stars1],
)
sp.draw_milky_way()
sp.draw_colorbar(label="Average g-r color")
plt.show()

Looking at the average g-r color of the stars shows a different perspective on the Galaxy.

What is going on here?

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

sp = skyproj.McBrydeSkyproj(ax=ax)
# We are zooming in on the region with the main part of the DP2 catalog.
hpxmap, _, _, _, _ = sp.draw_hpxbin(
    ra[stars1], 
    dec[stars1], 
    nest=True, 
    lon_range=[-130, -100], 
    lat_range=[-35, -10], 
    C=gmr[stars1], 
    nside=1024,
)
sp.draw_milky_way()
sp.draw_colorbar(label="Average g-r color")
plt.show()

Above we have a zoom in on the Northern part of the bulge, at higher resolution.

In [ ]:
sfd = SFDQuery()
# Sorry about the warning.

# Let's get the same valid positions as we have data for easier visualization
ra, dec = hpgeom.pixel_to_angle(hpgeom.npixel_to_nside(len(hpxmap)), np.where(hpxmap > 0.0)[0])
# And load the EBV values
ebv = sfd(SkyCoord(ra, dec, unit="degree", frame="icrs"))


In [ ]:
ebv_submap = hpxmap.copy()
ebv_submap[hpxmap > 0.0] = ebv

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

sp = skyproj.McBrydeSkyproj(ax=ax)
# We are zooming in on the region with the main part of the DP2 catalog.
sp.draw_hpxmap(
    ebv_submap,
    nest=True, 
    lon_range=[-130, -100], 
    lat_range=[-35, -10], 
)
sp.draw_milky_way()
sp.draw_colorbar(label="SFD98 E(B-V)")
plt.show()